# 07 · Exportar modelo estrella para Power BI
Se escribe en `DATA_PROCESSED` (disco D:). Tablas: `fact_episodio`, `fact_etapa`, `fact_ocupacion_horaria`,
`dim_fecha`, `dim_diagnostico`, `pronostico`, `escenarios`, `shap_importancia`, `perfil_dx`.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src import datos, flow
from src.config import FIGURES, DATA_INTERIM, DATA_PROCESSED, ETAPAS
pd.set_option("display.max_columns", 50); pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid", context="notebook")
def guardar(nombre):
    plt.tight_layout(); plt.savefig(FIGURES / f"{nombre}.png", dpi=150, bbox_inches="tight"); plt.show()


In [2]:
et = datos.etapas(); ep = datos.episodios(et)
cols = ["episodio_id", "person_id", "inicio", "fin", "los_h", "tipo_episodio", "tipo_detalle", "los_atipica", "hospitalizado", "paso_triage", "h_triage",
        "h_atencion_urg", "h_hospitalizacion", "h_hasta_hosp", "h_hasta_primer_lab", "n_areas_urg", "n_labs", "n_dx", "n_proc",
        "edad", "grupo_edad", "sexo", "hora_llegada", "dia_semana", "franja", "festivo", "episodios_previos",
        "reconsulta_72h", "ocupacion_urg_llegada", "dx_principal"]
f_ep = ep[cols].assign(fecha=ep.inicio.dt.date)
f_et = et[["episodio_id", "orden", "etapa", "inicio", "fin", "duracion_h", "espera_previa_h", "etapa_anterior"]]
dim_fecha = pd.DataFrame({"fecha": pd.date_range("2023-01-01", "2026-04-30")})
dim_fecha = dim_fecha.assign(anio=dim_fecha.fecha.dt.year, mes=dim_fecha.fecha.dt.month, nombre_mes=dim_fecha.fecha.dt.strftime("%b"),
    dia_semana=dim_fecha.fecha.dt.dayofweek, trimestre=dim_fecha.fecha.dt.quarter)
dim_dx = ep.groupby(["dx_principal", "dx_principal_nombre"]).size().rename("episodios").reset_index()
salidas = {"fact_episodio": f_ep, "fact_etapa": f_et, "dim_fecha": dim_fecha, "dim_diagnostico": dim_dx,
           "fact_ocupacion_horaria": pd.read_parquet(DATA_INTERIM / "ocupacion_horaria.parquet")}
for n in ["pronostico", "escenarios", "shap_importancia", "perfil_dx", "variantes", "transiciones"]:
    p = DATA_INTERIM / f"{n}.parquet"
    if p.exists(): salidas[n] = pd.read_parquet(p)
for n, df in salidas.items():
    df.to_csv(DATA_PROCESSED / f"{n}.csv", index=False, encoding="utf-8-sig")
    print(f"✔ {n:<24} {len(df):>10,} filas")

✔ fact_episodio               694,509 filas


✔ fact_etapa                1,572,702 filas
✔ dim_fecha                     1,216 filas
✔ dim_diagnostico               2,677 filas
✔ fact_ocupacion_horaria       29,016 filas
✔ pronostico                      120 filas
✔ escenarios                        8 filas
✔ shap_importancia                 54 filas
✔ perfil_dx                        60 filas
✔ variantes                     1,307 filas
✔ transiciones                     12 filas
